Make predictions for a simple regulatory constraint model on human chromosome 22.

In [1]:
import time
import bgshr
import numpy as np
import pandas
import matplotlib.pylab as plt

In [14]:
# The lookup table, a .csv file holding background selection predictions from moments++.
lookup_tbl_file = "data/lookup_tbl_equilibrium.csv.gz"

# A recombination map file in hapmap format, with columns "Position(bp)" and "Rate(cM/Mb)".
rec_map_file = "data/YRI_recombination_map_hapmap_format_hg38_chr_22.txt.gz"

# A BEDGRAPH-style file specifying mutation rates in uniform genomic intervals.
mut_tbl_file = "data/roulette_tbl_chr22.csv.gz"

# BED files holding start/end positions of constrained elements. In this example, we consider
# protein-coding sequence (CDS) and inferred promoter elements.
element_files = [
    "data/cds_merged_chr22.bed.gz",
    "data/promoters_chr22.bed.gz"]

# Specify the target Ne for prediction.
Ne = 30000

# Specify an unlinked B-value- a reduction in diversity due to purifying selection on unlinked sites
# (other chromosomes)
B_unlinked = 0.95

# Parameters of Gamma distributions of fitness effects (DFE), specified with dictionaries
# in an order matching `element_files`.
# Here we use the Kim (2017) DFE, which was inferred for nonsynonymous CDS sites
# for CDS annotations. We place 30.2% of the DFE mass on neutrality (s = 0) to model synonymous mutations.
# For promoter elements, we use the DFE from Barroso et al (2026).
dfes = [
    {"type": "gamma_neutral", "shape": 0.215, "scale": 0.0234, "p_neu": 0.302},
    {"type": "gamma", "shape": 0.111, "scale": 0.000500}]

# Set up a grid of focal sites, spaced every `spacing` bp along chromosome 22.
L = 50818468
spacing = 10000
xs = np.arange(spacing // 2, L, spacing // 2)

In [15]:
# Load and scale the lookup table to the target Ne. If `Ne0` is the original value built
# into the table, each physical parameter (s, u, r) is scaled by Ne0 / Ne.
df = pandas.read_csv(lookup_tbl_file)
df = bgshr.Util.scale_lookup_table(df, Ne)

# Scaling to an Ne larger than Ne0 (here 1e4) results in a grid of recombination fractions `r` 
# which does not reach 0.5: here we introduce dummy lines which restore large recombination 
# fractions to the table following scaling.
df = bgshr.Util.fill_in_lookup_table(df)

# Extend lookup table s-coefficient grid down to -1 using classic BGS theory.
min_s = np.min(df["s"])
ss_extend = -np.logspace(0, np.log10(-min_s), 17)[:-1]
df = bgshr.ClassicBGS.extend_lookup_table(df, ss_extend)

# Add a column with genetic map distances in Morgans to the table and get splines. This unit
# conversion simplifies distance calculations from the recombination map.
df = bgshr.Util.convert_lookup_table_to_morgans(df)
splines = bgshr.Util.generate_linear_splines(df)[2]

In [16]:
# Load the recombination map
rmap = bgshr.Util.load_recombination_map(rec_map_file, L)

# Load the mutation rate table
mut_df = pandas.read_csv(mut_tbl_file)
columns = mut_df.columns
tbl_intervals = np.stack((mut_df[columns[1]], 
    mut_df[columns[2]]), axis=1).astype(np.int64)
values = np.array(mut_df[columns[4]])

# Use mutation table information to build a site mutation map and apply a mask to it
# (here we do not have an accessibility mask, so no sites are actually masked).
umap = bgshr.Util.build_site_map(tbl_intervals, values)
umap = np.ma.array(umap, mask=np.zeros(len(umap)))

# Load elements from BED files and purge any promoter sites which overlap CDS sites
# using `resolve_elements`. Classes of elements which are lower-indexed have higher
# priority in resolution.
elements = [bgshr.Util.read_bedfile(f) for f in element_files]
elements = bgshr.Util.resolve_elements(elements, verbose=True)

# We track constrained sites in uniform windows. This reduces compute time when elements
# are numerous. Here we set up windows and calculate the mutation rate assigned to each
# element class (CDS and promoters), within each window, then drop empty windows.
ws = 1000
windows = np.stack([np.arange(0, L - ws, ws), np.arange(ws, L, ws)], axis=1)
U_arrs = [bgshr.Util.compute_window_mutation_rates(windows, elems, umap)[0]
          for elems in elements]
windows, U_arrs = bgshr.Util.filter_empty_windows(windows, U_arrs)

[26-08-10 11:29:09] removed 159 redundant sites from element class 1


In [17]:
# Run prediction. We apply two rounds of interference correction, which repeats B prediction
# with local parameters (s, u, r) rescaled by the B-value calculated in the prior round of
# prediction, to approximate interference between constrained sites.
B_arrs = bgshr.Predict.interference_Bvals(
    xs,
    splines,
    windows=windows,
    U_arrs=U_arrs,
    rmap=rmap,
    max_dist=0.1,
    dfes=dfes,
    n_corrs=5,
    B_unlinked=B_unlinked,
    chunk_size=100,
    n_cores=8,
    verbose=True
)

[26-08-10 11:29:27] completed initial B-prediction
[26-08-10 11:29:47] completed interference correction 1
[26-08-10 11:30:07] completed interference correction 2
[26-08-10 11:30:28] completed interference correction 3
[26-08-10 11:30:48] completed interference correction 4
[26-08-10 11:31:08] completed interference correction 5


In [ ]:
# Plot the B-landscape from each round of interference correction.
fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
ax.plot(xs, B_arrs[0], label="round 0")
ax.plot(xs, B_arrs[1], label="round 1")
ax.plot(xs, B_arrs[5], label="round 5")
ax.legend(framealpha=0)
ax.set_title("Background selection landscape on chromosome 22 (CDS/promoter model)")
ax.set_ylabel("$B$")
ax.set_xlabel("Position (Mb)")
ax.set_xticks([0, 1e7, 2e7, 3e7, 4e7, 5e7], [0, 10, 20, 30, 40, 50])